In [1]:
# 🚀 Install PyTorch Nightly with CUDA 12.8 for RTX 5090 Support
# This will install the latest PyTorch with proper RTX 5090 kernel support

import subprocess
import sys

def install_pytorch_nightly():
    """Install PyTorch nightly build with CUDA 12.8 support"""
    
    print("🔄 Installing PyTorch Nightly with CUDA 12.8 support for RTX 5090...")
    print("⚠️  This may take several minutes to download and install.")
    print("=" * 80)
    
    # The command to install PyTorch nightly with CUDA 12.8
    install_command = [
        sys.executable, "-m", "pip", "install", "--pre", 
        "torch", "torchvision", 
        "--index-url", "https://download.pytorch.org/whl/nightly/cu128"
    ]
    
    try:
        # Run the installation
        print("Running command:")
        print(" ".join(install_command))
        print("\n" + "="*50)
        
        result = subprocess.run(
            install_command,
            capture_output=True,
            text=True,
            timeout=600  # 10 minute timeout
        )
        
        # Print output
        if result.stdout:
            print("📥 Installation Output:")
            print(result.stdout)
        
        if result.stderr:
            print("⚠️  Installation Messages:")
            print(result.stderr)
        
        if result.returncode == 0:
            print("\n✅ PyTorch nightly installation completed successfully!")
            print("🔄 Please RESTART your kernel after installation completes")
            print("📝 After restarting, check PyTorch version with: torch.__version__")
        else:
            print(f"\n❌ Installation failed with return code: {result.returncode}")
            
    except subprocess.TimeoutExpired:
        print("⏰ Installation timed out (took longer than 10 minutes)")
        print("💡 Try running the command manually in terminal:")
        print("pip install --pre torch torchvision --index-url https://download.pytorch.org/whl/nightly/cu128")
        
    except Exception as e:
        print(f"❌ Installation error: {e}")
        print("💡 Try running manually in terminal:")
        print("pip install --pre torch torchvision --index-url https://download.pytorch.org/whl/nightly/cu128")

# Run the installation
install_pytorch_nightly()

print("\n" + "="*80)
print("🎯 NEXT STEPS AFTER INSTALLATION:")
print("1. RESTART your Jupyter kernel (Kernel → Restart)")
print("2. Run: import torch; print(torch.__version__)")
print("3. Test GPU: torch.cuda.is_available()")
print("4. Try loading your model again - RTX 5090 should now work!")
print("="*80)

🔄 Installing PyTorch Nightly with CUDA 12.8 support for RTX 5090...
⚠️  This may take several minutes to download and install.
Running command:
c:\Python314\python.exe -m pip install --pre torch torchvision --index-url https://download.pytorch.org/whl/nightly/cu128

📥 Installation Output:
Looking in indexes: https://download.pytorch.org/whl/nightly/cu128


✅ PyTorch nightly installation completed successfully!
🔄 Please RESTART your kernel after installation completes
📝 After restarting, check PyTorch version with: torch.__version__

🎯 NEXT STEPS AFTER INSTALLATION:
1. RESTART your Jupyter kernel (Kernel → Restart)
2. Run: import torch; print(torch.__version__)
3. Test GPU: torch.cuda.is_available()
4. Try loading your model again - RTX 5090 should now work!


In [2]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import re
import os
import glob
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report, mean_absolute_error, mean_squared_error
import numpy as np
import warnings
import gc
import matplotlib.pyplot as plt
import seaborn as sns

# Check GPU availability
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU device:", torch.cuda.get_device_name(0))
    print("Number of GPUs:", torch.cuda.device_count())
    print("Current GPU:", torch.cuda.current_device())
    print("GPU memory allocated:", torch.cuda.memory_allocated(0) / 1024**3, "GB")
    print("GPU memory reserved:", torch.cuda.memory_reserved(0) / 1024**3, "GB")
else:
    print("⚠️ WARNING: No GPU detected! Running on CPU will be extremely slow.")
    print("Please ensure:")
    print("  1. You have an NVIDIA GPU installed")
    print("  2. CUDA drivers are installed")
    print("  3. PyTorch CUDA version matches your CUDA driver")

# Disable tokenizer parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Paths
image_path = r"D:\Dementia-Thesis - Don't access without permission\Cookie-Theft-stimulus.png"
transcription_dir = r"D:\Dementia-Thesis - Don't access without permission\cha_par_only"
ground_truth_path = r"D:\Dementia-Thesis - Don't access without permission\2_classes.csv"
output_csv_path = r"D:\Dementia-Thesis - Don't access without permission\qwen_zero_shot.csv"

# ============================================================================
# DEBUG MODE - SET TO True TO SEE MODEL OUTPUTS
# ============================================================================
DEBUG_MODE = True  # Set to False to disable verbose output after debugging
DEBUG_SAMPLES = 10  # Number of samples to show full output for

# ============================================================================
# AGGRESSIVE MEMORY CLEANUP BEFORE MODEL LOADING
# ============================================================================
print("\n" + "="*80)
print("FREEING MEMORY BEFORE MODEL LOADING...")
print("="*80)

# Delete any existing model/processor objects if they exist in memory
if 'model' in dir():
    print("Deleting existing model...")
    del model
if 'processor' in dir():
    print("Deleting existing processor...")
    del processor

# Force garbage collection
gc.collect()

# Clear CUDA cache if available
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"GPU memory after cleanup: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB allocated")
    print(f"GPU memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
else:
    print("CPU mode - no GPU cache to clear")

print("✅ Memory cleanup complete!\n")

# Load ground truth
print("Loading ground truth data...")
df_gt = pd.read_csv(ground_truth_path)

# Normalize ground truth labels to lowercase for consistent matching
if 'dx' in df_gt.columns:
    df_gt['dx'] = df_gt['dx'].apply(lambda x: str(x).lower() if pd.notna(x) else None)

print(f"Ground truth loaded: {len(df_gt)} total records")

# Filter to only records WITH valid dx labels
df_gt_valid = df_gt[df_gt['dx'].notna()].copy()
print(f"Records with valid dx labels: {len(df_gt_valid)}")
print(f"Records WITHOUT dx labels (will be skipped): {len(df_gt) - len(df_gt_valid)}")
print(f"Unique dx labels: {df_gt_valid['dx'].unique()}")

# Get all .cha files
cha_files_all = sorted(glob.glob(os.path.join(transcription_dir, "*.cha")))
print(f"\nFound {len(cha_files_all)} total .cha files")

# Filter to only process files with valid dx labels
valid_ids = set(df_gt_valid['id'].astype(str))
cha_files = [f for f in cha_files_all if os.path.splitext(os.path.basename(f))[0] in valid_ids]

skipped_count = len(cha_files_all) - len(cha_files)
print(f"Files to process (with valid dx): {len(cha_files)}")
print(f"Files skipped (no dx label): {skipped_count}")

# Load image (shared across all analyses)
image = Image.open(image_path)

# Load model and processor - FULL MODEL WITHOUT QUANTIZATION
print("\n" + "="*80)
print("LOADING FULL MODEL WITHOUT QUANTIZATION FOR ZERO-SHOT CLASSIFICATION...")
print("="*80)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Loading processor...")
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
print("✅ Processor loaded")

try:
    print("Loading full model on GPU (no quantization)...")
    model = AutoModelForImageTextToText.from_pretrained(
        "Qwen/Qwen2.5-VL-3B-Instruct",
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    print("✅ Full model loaded successfully on GPU!")
    if torch.cuda.is_available():
        print(f"GPU memory after model load: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB allocated")
        print(f"GPU memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
except Exception as e:
    print(f"❌ GPU loading failed: {str(e)}")
    print("🔄 Falling back to CPU...")
    torch.cuda.empty_cache()
    model = AutoModelForImageTextToText.from_pretrained(
        "Qwen/Qwen2.5-VL-3B-Instruct",
        torch_dtype=torch.float16,
        device_map="cpu",
        low_cpu_mem_usage=True
    )
    print("✅ Model loaded successfully on CPU! (This will be slower)")

print("Model loading complete! Using FULL MODEL without quantization.")
print("="*80 + "\n")

results = []

print("\n" + "="*80)
print("SANITY CHECK: Sampling first 3 transcriptions...")
print("="*80)
for i, cha_file in enumerate(cha_files[:3]):
    file_name = os.path.basename(cha_file)
    print(f"\nFile {i+1}: {file_name}")
    with open(cha_file, 'r', encoding='utf-8', errors='replace') as f:
        sample = f.read().strip()
    print(f"Length: {len(sample)} chars, {len(sample.split())} words")
    print(f"Content preview:\n{sample[:400]}...")
    print("-" * 80)

print("\n" + "="*80)
print("Starting full processing...")
print("="*80 + "\n")

processing_diagnosis_counts = {'control': 0, 'probablead': 0, 'unknown': 0, 'error': 0}
processing_count = 0

for cha_file in tqdm(cha_files, desc="Processing files"):
    file_name = os.path.basename(cha_file)
    file_id = os.path.splitext(file_name)[0]
    try:
        with open(cha_file, 'r', encoding='utf-8', errors='replace') as f:
            transcription = f.read().strip()
        if len(transcription.strip()) < 10:
            results.append({
                'id': file_id,
                'file': file_name,
                'predicted_diagnosis': 'error',
                'predicted_mmse': -1,
                'reasoning': 'Transcription too short or empty',
                'transcription_length': len(transcription),
                'raw_model_output': ''
            })
            processing_diagnosis_counts['error'] += 1
            continue
        messages_diagnosis = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": f"""Analyze this Cookie Theft picture description for cognitive assessment.

DESCRIPTION:
{transcription}

Classify as:
- 'control' = normal cognitive function
- 'probablead' = probable Alzheimer's disease

Provide your assessment:

DIAGNOSIS: control OR probablead
MMSE_SCORE: [0-30]
REASONING: [Brief explanation]"""}
                ]
            }
        ]
        text_prompt_diagnosis = processor.apply_chat_template(messages_diagnosis, add_generation_prompt=True, tokenize=False)
        inputs_diagnosis = processor(text=[text_prompt_diagnosis], images=[image], padding=True, return_tensors="pt")
        inputs_diagnosis = {k: v.to(model.device) for k, v in inputs_diagnosis.items()}
        input_length = inputs_diagnosis["input_ids"].shape[-1]
        with torch.no_grad():
            outputs_diagnosis = model.generate(
                **inputs_diagnosis, 
                max_new_tokens=250,
                do_sample=True,
                temperature=0.5,  # Lower for more consistent/less biased output
                top_p=0.85,
                top_k=40,
                repetition_penalty=1.15,
                pad_token_id=processor.tokenizer.eos_token_id
            )
        del inputs_diagnosis
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        diagnosis_result = processor.decode(outputs_diagnosis[0][input_length:], skip_special_tokens=True)
        if DEBUG_MODE and processing_count < DEBUG_SAMPLES:
            print(f"\n{'='*80}")
            print(f"🔍 DEBUG OUTPUT #{processing_count + 1} - File: {file_id}")
            print(f"{'='*80}")
            print(f"TRANSCRIPTION (first 200 chars): {transcription[:200]}...")
            print(f"\n{'─'*80}")
            print(f"RAW MODEL OUTPUT:")
            print(f"{'─'*80}")
            print(diagnosis_result)
            print(f"{'='*80}\n")
        diagnosis = None
        diagnosis_match = re.search(r'DIAGNOSIS:\s*(\w+)', diagnosis_result, re.IGNORECASE)
        if diagnosis_match:
            extracted = diagnosis_match.group(1).lower()
            if 'probable' in extracted or extracted == 'probablead':
                diagnosis = 'probablead'
            elif extracted == 'control':
                diagnosis = 'control'
        if not diagnosis:
            if re.search(r'\bprobable\s*ad\b|\bprobablead\b', diagnosis_result, re.IGNORECASE):
                diagnosis = 'probablead'
            elif re.search(r'\bcontrol\b', diagnosis_result, re.IGNORECASE):
                diagnosis = 'control'
        if not diagnosis:
            dementia_keywords = re.search(r'\balzheimer|\bdementia\b|\b(?:probable|possible)\s*(?:ad|a\.d\.)\b', diagnosis_result, re.IGNORECASE)
            healthy_keywords = re.search(r'\bhealthy\b|\bnormal\b|\bno\s+(?:dementia|impairment)\b|\bintact\b', diagnosis_result, re.IGNORECASE)
            if dementia_keywords and not healthy_keywords:
                diagnosis = 'probablead'
            elif healthy_keywords and not dementia_keywords:
                diagnosis = 'control'
            elif dementia_keywords and healthy_keywords:
                dementia_pos = dementia_keywords.start()
                healthy_pos = healthy_keywords.start()
                diagnosis = 'probablead' if dementia_pos < healthy_pos else 'control'
        if not diagnosis:
            first_300 = diagnosis_result[:300].lower()
            if re.search(r'\bminimal\s+cognitive\s+impairment\b|\bmci\b|\bmild\s+cognitive\s+impairment\b', diagnosis_result, re.IGNORECASE):
                if re.search(r'\bvery\s+mild\b|\bminimal\b|\bborderline\b', first_300):
                    diagnosis = 'control'
                else:
                    diagnosis = 'probablead'
        if not diagnosis:
            first_part = diagnosis_result[:300].lower()
            dementia_score = sum([
                'alzheimer' in first_part,
                'dementia' in first_part,
                'impaired' in first_part,
                'impairment' in first_part,
                'deficit' in first_part,
                'cognitive decline' in first_part,
                'deteriorat' in first_part,
                'abnormal' in first_part
            ])
            control_score = sum([
                'healthy' in first_part,
                'normal' in first_part,
                'intact' in first_part,
                'no impair' in first_part,
                'no deficit' in first_part,
                'control' in first_part,
                'unimpaired' in first_part
            ])
            if dementia_score > control_score and dementia_score > 0:
                diagnosis = 'probablead'
            elif control_score > dementia_score and control_score > 0:
                diagnosis = 'control'
        if not diagnosis:
            diagnosis = 'unknown'
        reasoning_match = re.search(r'REASONING:(.+?)(?:\n\n|\Z)', diagnosis_result, re.IGNORECASE | re.DOTALL)
        # IMPROVED MMSE extraction
        mmse_score = -1
        mmse_text = None
        mmse_patterns = [
            r'MMSE[_\s]*SCORE[:\s]+([^\n]+)',
            r'MMSE[:\s]+([^\n]+)',
            r'score[:\s]+([^\n]+)',
        ]
        for pattern in mmse_patterns:
            mmse_match = re.search(pattern, diagnosis_result, re.IGNORECASE)
            if mmse_match:
                mmse_text = mmse_match.group(1).strip()
                break
        if not mmse_text:
            mmse_text = None
            after_mmse = re.search(r'(mmse|score)[^\d]*(.*)', diagnosis_result, re.IGNORECASE)
            if after_mmse:
                mmse_text = after_mmse.group(2)[:100].strip()
        if mmse_text:
            mmse_text_lower = mmse_text.lower()
            if any(x in mmse_text_lower for x in ["not applicable", "n/a", "requires formal", "not available", "unknown"]):
                mmse_score = -1
            elif re.match(r'\d+\s*[-–]\s*\d+', mmse_text):
                nums = re.findall(r'\d+', mmse_text)
                if len(nums) == 2:
                    mmse_score = int(round((int(nums[0]) + int(nums[1])) / 2))
            elif re.match(r'(likely )?below (\d+)', mmse_text_lower):
                x = re.search(r'below (\d+)', mmse_text_lower)
                if x:
                    mmse_score = int(x.group(1)) - 1
            elif re.match(r'(likely )?above (\d+)', mmse_text_lower):
                x = re.search(r'above (\d+)', mmse_text_lower)
                if x:
                    mmse_score = int(x.group(1)) + 1
            elif re.match(r'(\d+)\s*/\s*(\d+)', mmse_text):
                nums = re.findall(r'\d+', mmse_text)
                if len(nums) >= 1:
                    val = int(nums[0])
                    if 0 <= val <= 30:
                        mmse_score = val
            elif re.match(r'(likely|approximate|about|around) (\d+)', mmse_text_lower):
                x = re.search(r'(likely|approximate|about|around) (\d+)', mmse_text_lower)
                if x:
                    mmse_score = int(x.group(2))
            elif re.match(r'\d+', mmse_text):
                val = int(re.match(r'\d+', mmse_text).group(0))
                if 0 <= val <= 30:
                    mmse_score = val
            else:
                nums = re.findall(r'\d+', mmse_text)
                for num_str in nums:
                    num = int(num_str)
                    if 0 <= num <= 30:
                        mmse_score = num
                        break
        else:
            all_numbers = re.findall(r'\b(\d+)\b', diagnosis_result)
            for num_str in all_numbers:
                try:
                    num = int(num_str)
                    if 0 <= num <= 30:
                        mmse_score = num
                        break
                except ValueError:
                    pass
        reasoning = reasoning_match.group(1).strip() if reasoning_match else diagnosis_result[:200]
        processing_diagnosis_counts[diagnosis] += 1
        processing_count += 1
        results.append({
            'id': file_id,
            'file': file_name,
            'predicted_diagnosis': diagnosis,
            'predicted_mmse': mmse_score,
            'reasoning': reasoning,
            'transcription_length': len(transcription),
            'raw_model_output': diagnosis_result if DEBUG_MODE else ''
        })
        if not DEBUG_MODE or processing_count > DEBUG_SAMPLES:
            print(f"✅ {file_id}: {diagnosis} (MMSE: {mmse_score if mmse_score != -1 else 'N/A'})")
        if processing_count % 20 == 0:
            print(f"\n📊 Running distribution after {processing_count} files:")
            for dx, count in processing_diagnosis_counts.items():
                pct = (count / processing_count * 100) if processing_count > 0 else 0
                print(f"  {dx}: {count} ({pct:.1f}%)")
            print()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        print(f"❌ ERROR processing {file_name}: {str(e)}")
        print(f"Marking as 'error' with MMSE=-1")
        import traceback
        traceback.print_exc()
        results.append({
            'id': file_id,
            'file': file_name,
            'predicted_diagnosis': 'error',
            'predicted_mmse': -1,
            'reasoning': f'Processing error: {str(e)[:100]}',
            'transcription_length': 0,
            'raw_model_output': ''
        })
        processing_diagnosis_counts['error'] += 1
        continue
print(f"\n{'='*80}")
print(f"PROCESSING COMPLETE!")
print(f"Total files processed: {len(results)}")
print('='*80)
# ...rest of the code remains unchanged...

c:\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.11.0.dev20251223+cu128
CUDA available: True
CUDA version: 12.8
GPU device: NVIDIA GeForce RTX 5090
Number of GPUs: 1
Current GPU: 0
GPU memory allocated: 0.0 GB
GPU memory reserved: 0.0 GB

FREEING MEMORY BEFORE MODEL LOADING...
GPU memory after cleanup: 0.00 GB allocated
GPU memory reserved: 0.00 GB
✅ Memory cleanup complete!

Loading ground truth data...
Ground truth loaded: 498 total records
Records with valid dx labels: 498
Records WITHOUT dx labels (will be skipped): 0
Unique dx labels: ['probablead' 'control']

Found 552 total .cha files
Files to process (with valid dx): 498
Files skipped (no dx label): 54

LOADING FULL MODEL WITHOUT QUANTIZATION FOR ZERO-SHOT CLASSIFICATION...
Loading processor...


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


✅ Processor loaded
Loading full model on GPU (no quantization)...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.49s/it]


...

✅ 709-2: probablead (MMSE: N/A)


Processing files: 100%|█████████▉| 497/498 [1:19:10<00:10, 10.21s/it]

✅ 711-0: probablead (MMSE: 28)


Processing files: 100%|██████████| 498/498 [1:19:21<00:00,  9.56s/it]

✅ 714-0: probablead (MMSE: 25)

PROCESSING COMPLETE!
Total files processed: 498


In [3]:
# ============================================================================
# CALCULATE METRICS AND SAVE RESULTS TO CSV
# ============================================================================

# Create DataFrame from results
df_results = pd.DataFrame(results)

# Count error/unknown predictions
error_count = (df_results['predicted_diagnosis'] == 'error').sum()
unknown_count = (df_results['predicted_diagnosis'] == 'unknown').sum()
valid_count = len(df_results) - error_count - unknown_count

print(f"\n{'='*80}")
print("RESULTS SUMMARY")
print('='*80)
print(f"\nTotal predictions: {len(df_results)}")
print(f"  Valid predictions: {valid_count}")
print(f"  Unknown predictions: {unknown_count}")
print(f"  Error predictions: {error_count}")

# Print diagnosis distribution (valid only)
valid_results = df_results[~df_results['predicted_diagnosis'].isin(['error', 'unknown'])]
if len(valid_results) > 0:
    print(f"\nDiagnosis distribution (valid predictions):")
    diagnosis_counts = valid_results['predicted_diagnosis'].value_counts()
    for dx, count in diagnosis_counts.items():
        pct = (count / len(valid_results) * 100)
        print(f"  {dx}: {count} ({pct:.1f}%)")

# MMSE statistics
valid_mmse = df_results[df_results['predicted_mmse'] >= 0]['predicted_mmse']
if len(valid_mmse) > 0:
    print(f"\nMMSE score statistics (valid predictions):")
    print(f"  Mean:   {valid_mmse.mean():.2f}")
    print(f"  Median: {valid_mmse.median():.2f}")
    print(f"  Std:    {valid_mmse.std():.2f}")
    print(f"  Range:  {valid_mmse.min()} - {valid_mmse.max()}")

# ============================================================================
# MERGE WITH GROUND TRUTH AND CALCULATE METRICS
# ============================================================================

print(f"\n{'='*80}")
print("CALCULATING METRICS")
print('='*80)

# Merge predictions with ground truth
df_merged = df_results.merge(df_gt_valid, on='id', how='inner')
print(f"\nMerged {len(df_merged)} records with ground truth")

if len(df_merged) > 0:
    # Classification metrics (error/unknown count as wrong)
    y_true = df_merged['dx'].values
    y_pred = df_merged['predicted_diagnosis'].values
    
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(df_merged)
    accuracy_all = correct_predictions / total_predictions
    
    error_unknown_count = df_merged['predicted_diagnosis'].isin(['error', 'unknown']).sum()
    
    print(f"\n--- CLASSIFICATION METRICS (error/unknown count as WRONG) ---")
    print(f"Total with ground truth: {total_predictions}")
    print(f"Correct predictions: {correct_predictions}")
    print(f"Wrong predictions: {total_predictions - correct_predictions}")
    print(f"  - of which error/unknown: {error_unknown_count}")
    print(f"Accuracy (ALL): {accuracy_all:.4f}")
    
    # Calculate standard metrics only on valid predictions
    df_valid = df_merged[~df_merged['predicted_diagnosis'].isin(['error', 'unknown'])].copy()
    
    if len(df_valid) > 0:
        y_true_valid = df_valid['dx'].values
        y_pred_valid = df_valid['predicted_diagnosis'].values
        
        accuracy_valid = accuracy_score(y_true_valid, y_pred_valid)
        precision, recall, f1, support = precision_recall_fscore_support(
            y_true_valid, y_pred_valid, 
            average='binary', 
            pos_label='probablead',
            zero_division=0
        )
        
        # Confusion Matrix
        cm = confusion_matrix(y_true_valid, y_pred_valid, labels=['probablead', 'control'])
        tp, fn, fp, tn = cm.ravel()
        
        # Specificity and Sensitivity
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        print(f"\n--- METRICS ON VALID PREDICTIONS ONLY ---")
        print(f"Valid predictions evaluated: {len(df_valid)}")
        print(f"Accuracy:    {accuracy_valid:.4f}")
        print(f"Precision:   {precision:.4f}")
        print(f"Recall:      {recall:.4f}")
        print(f"F1-Score:    {f1:.4f}")
        print(f"Sensitivity: {sensitivity:.4f}")
        print(f"Specificity: {specificity:.4f}")
        
        print("\nConfusion Matrix (valid predictions):")
        print(f"                  Predicted")
        print(f"Actual       probableAD  control")
        print(f"probableAD     {tp:5d}    {fn:5d}")
        print(f"control        {fp:5d}    {tn:5d}")
    else:
        accuracy_valid = precision = recall = f1 = sensitivity = specificity = 0
        tp = tn = fp = fn = 0
    
    # MMSE Metrics
    records_with_mmse = df_merged[df_merged['mmse'].notna()].copy()
    
    if len(records_with_mmse) > 0:
        correct_mmse = (records_with_mmse['mmse'] == records_with_mmse['predicted_mmse']).sum()
        total_mmse = len(records_with_mmse)
        error_mmse = (records_with_mmse['predicted_mmse'] == -1).sum()
        
        print(f"\n--- MMSE METRICS (error/-1 counts as WRONG) ---")
        print(f"Total with ground truth MMSE: {total_mmse}")
        print(f"Exact matches: {correct_mmse}")
        print(f"Wrong predictions: {total_mmse - correct_mmse}")
        print(f"  - of which error/unknown (-1): {error_mmse}")
        
        # MAE/RMSE only on valid predictions
        df_valid_mmse = records_with_mmse[records_with_mmse['predicted_mmse'] >= 0]
        
        if len(df_valid_mmse) > 0:
            y_true_mmse = df_valid_mmse['mmse'].values
            y_pred_mmse = df_valid_mmse['predicted_mmse'].values
            
            mae = mean_absolute_error(y_true_mmse, y_pred_mmse)
            mse = mean_squared_error(y_true_mmse, y_pred_mmse)
            rmse = np.sqrt(mse)
            
            print(f"\nRegression metrics (valid predictions, excluding -1):")
            print(f"MAE (Mean Absolute Error): {mae:.4f}")
            print(f"MSE (Mean Squared Error):  {mse:.4f}")
            print(f"RMSE (Root MSE):           {rmse:.4f}")
            print(f"Samples evaluated:         {len(df_valid_mmse)}")
        else:
            mae = mse = rmse = np.nan
    else:
        mae = mse = rmse = np.nan
        correct_mmse = total_mmse = error_mmse = 0
    
    # Create metrics summary rows
    metrics_summary = [
        {'id': '', 'file': '--- METRICS SUMMARY ---', 'predicted_diagnosis': '', 'predicted_mmse': np.nan, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'METRIC', 'file': 'Accuracy (ALL, error=wrong)', 'predicted_diagnosis': '', 'predicted_mmse': accuracy_all, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'METRIC', 'file': 'Accuracy (valid only)', 'predicted_diagnosis': '', 'predicted_mmse': accuracy_valid, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'METRIC', 'file': 'Precision', 'predicted_diagnosis': '', 'predicted_mmse': precision, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'METRIC', 'file': 'Recall', 'predicted_diagnosis': '', 'predicted_mmse': recall, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'METRIC', 'file': 'F1-Score', 'predicted_diagnosis': '', 'predicted_mmse': f1, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'METRIC', 'file': 'Sensitivity', 'predicted_diagnosis': '', 'predicted_mmse': sensitivity, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'METRIC', 'file': 'Specificity', 'predicted_diagnosis': '', 'predicted_mmse': specificity, 'reasoning': '', 'transcription_length': np.nan},
        {'id': '', 'file': '', 'predicted_diagnosis': '', 'predicted_mmse': np.nan, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'CONFUSION', 'file': 'True Positives', 'predicted_diagnosis': '', 'predicted_mmse': int(tp), 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'CONFUSION', 'file': 'True Negatives', 'predicted_diagnosis': '', 'predicted_mmse': int(tn), 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'CONFUSION', 'file': 'False Positives', 'predicted_diagnosis': '', 'predicted_mmse': int(fp), 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'CONFUSION', 'file': 'False Negatives', 'predicted_diagnosis': '', 'predicted_mmse': int(fn), 'reasoning': '', 'transcription_length': np.nan},
        {'id': '', 'file': '', 'predicted_diagnosis': '', 'predicted_mmse': np.nan, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'MMSE', 'file': 'MAE', 'predicted_diagnosis': '', 'predicted_mmse': mae, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'MMSE', 'file': 'MSE', 'predicted_diagnosis': '', 'predicted_mmse': mse, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'MMSE', 'file': 'RMSE', 'predicted_diagnosis': '', 'predicted_mmse': rmse, 'reasoning': '', 'transcription_length': np.nan},
        {'id': '', 'file': '', 'predicted_diagnosis': '', 'predicted_mmse': np.nan, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'COUNT', 'file': 'Total Predictions', 'predicted_diagnosis': '', 'predicted_mmse': len(df_results), 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'COUNT', 'file': 'Valid DX Predictions', 'predicted_diagnosis': '', 'predicted_mmse': len(df_valid) if len(df_valid) > 0 else 0, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'COUNT', 'file': 'Error/Unknown DX', 'predicted_diagnosis': '', 'predicted_mmse': error_unknown_count, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'COUNT', 'file': 'Valid MMSE Predictions', 'predicted_diagnosis': '', 'predicted_mmse': len(df_valid_mmse) if len(records_with_mmse) > 0 and len(df_valid_mmse) > 0 else 0, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'COUNT', 'file': 'Error/Unknown MMSE (-1)', 'predicted_diagnosis': '', 'predicted_mmse': error_mmse, 'reasoning': '', 'transcription_length': np.nan},
    ]
    
    df_with_metrics = pd.concat([df_results, pd.DataFrame(metrics_summary)], ignore_index=True)
else:
    df_with_metrics = df_results

# ============================================================================
# SAVE RESULTS TO CSV
# ============================================================================

try:
    df_with_metrics.to_csv(output_csv_path, index=False)
    print(f"\n{'='*80}")
    print(f"✅ RESULTS SAVED TO CSV!")
    print(f"{'='*80}")
    print(f"Output file: {output_csv_path}")
    print(f"Total records saved: {len(df_with_metrics)}")
    print(f"  - Predictions: {len(df_results)}")
    print(f"  - Metrics summary: {len(metrics_summary) if len(df_merged) > 0 else 0}")
    print(f"\nNote: {skipped_count} files were skipped (no dx label in ground truth)")
    print('='*80)
except Exception as e:
    print(f"\n❌ ERROR saving CSV: {str(e)}")
    print("Results are still available in df_results DataFrame")

print("\n✅ Done!")


RESULTS SUMMARY

Total predictions: 498
  Valid predictions: 498
  Unknown predictions: 0
  Error predictions: 0

Diagnosis distribution (valid predictions):
  probablead: 328 (65.9%)
  control: 170 (34.1%)

MMSE score statistics (valid predictions):
  Mean:   23.73
  Median: 25.00
  Std:    4.38
  Range:  1 - 29

CALCULATING METRICS

Merged 498 records with ground truth

--- CLASSIFICATION METRICS (error/unknown count as WRONG) ---
Total with ground truth: 498
Correct predictions: 283
Wrong predictions: 215
  - of which error/unknown: 0
Accuracy (ALL): 0.5683

--- METRICS ON VALID PREDICTIONS ONLY ---
Valid predictions evaluated: 498
Accuracy:    0.5683
Precision:   0.5610
Recall:      0.7216
F1-Score:    0.6312
Sensitivity: 0.7216
Specificity: 0.4074

Confusion Matrix (valid predictions):
                  Predicted
Actual       probableAD  control
probableAD       184       71
control          144       99

--- MMSE METRICS (error/-1 counts as WRONG) ---
Total with ground truth MMS